In [8]:

from PIL import Image, ImageDraw
import pandas as pd
import cv2
import os
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
import glob
import torch
import argparse
import torch.nn.functional as F
import torch.nn as nn
from shapely.geometry import Polygon

def get_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

def get_iou_polygon(polygonA, polygonB):
    # This function would calculate the IoU between two polygons.
    polyA = Polygon([(polygonA[i], polygonA[i + 1]) for i in range(0, len(polygonA), 2)])
    polyB = Polygon([(polygonB[i], polygonB[i + 1]) for i in range(0, len(polygonB), 2)])
    iou = polyA.intersection(polyB).area / polyA.union(polyB).area
    return iou

def nms_classwise(df):
    def nms_single_class(df_class):
        elements = df_class.to_dict('records')
        elements.sort(key=lambda x: x['confidence'], reverse=True)
        selected_elements = []

        for current in elements:
            # Correctly using list comprehension for creating a list of vertices for the current polygon
            current_polygon = [val for i in range(4) for val in (current[f'x{i+1}'], current[f'y{i+1}'])]

            overlap = False
            for selected in selected_elements:
                # Similarly constructing the polygon vertices list for the selected element
                selected_polygon = [val for i in range(4) for val in (selected[f'x{i+1}'], selected[f'y{i+1}'])]

                if get_iou_polygon(current_polygon, selected_polygon) > 0.3:  # Using the corrected polygon IoU function
                    overlap = True
                    break

            if not overlap:
                selected_elements.append(current)

        return pd.DataFrame(selected_elements)

    grouped = df.groupby('label')
    nms_dfs = [nms_single_class(group) for _, group in grouped]
    return pd.concat(nms_dfs, ignore_index=True)

def get_symmetrical_angle_ranges_with_median(df):
    pivot = 90
    # Adjust all angles to be within 0-90 based on their symmetry around the pivot
    df['adjusted_angle'] = df['angle'].apply(lambda x: x if x <= pivot else 180 - x)
    bin_counts = np.histogram(df['adjusted_angle'], bins=[0, 15, 30, 45, 60, 75, 90])[0]
    top_two_bins = bin_counts.argsort()[-2:][::-1]  # Get indexes of the two highest bins
    total_count_top_two_bins = sum(bin_counts[top_two_bins])
    if total_count_top_two_bins / len(df) * 100 > 80:
        median_rotation = np.median(df['adjusted_angle'])
        return True, median_rotation
    else:
        return False, None


def get_yolo_boxes(image_path, cropped_img = None, dump_images=False, conf_thresh = .1, estimate_rotation = True, dump_txt = False):
    try:
        im = np.array(cropped_img)
        im = im[:, :, ::-1].copy()
    except:
        im = cv2.imread(image_path)
    pageheight, pagewidth= im.shape[0], im.shape[1]
    predictions = det_model(im, imgsz=800, conf=0.05, max_det=10000, verbose=False)
    xywhr =  predictions[0].obb.xywhr.tolist()
    xyxyxyxy = predictions[0].obb.xyxyxyxy.tolist()
    xyxy = predictions[0].obb.xyxy.tolist()
    classes = predictions[0].obb.cls.tolist()
    confidence = predictions[0].obb.conf.tolist()
    names = [
        "box", "table", "column", "header",
        "signature", "figure", "paragraph", "logo", "kv", "stamp"
    ] # Get class names
    name_to_id = {name: i for i, name in enumerate(names)}

    # Color codes for each label
    color_codes = {
        "box": (255, 0, 0),  # Blue
        "header": (255, 0, 0),  # Blue
        "table": (0, 255, 0),  # Green
        "column": (0, 255, 0),  # Green
        "logo": (0, 0, 255),  # Red
        "signature": (0, 255, 255),  # Yellow
        "stamp": (0, 0, 0),  # Black
        # Default color for other labels (optional)
        "default": (255, 255, 255)  # White
    }
    if dump_images:
        # Check if results directory exists, if not, create it
        if not os.path.exists('results'):
            os.makedirs('results')
        # Make a copy of the image to draw on
        im_copy = im.copy()
    df_data = []
    for i, a in enumerate(xywhr):
        centroid_x, centroid_y, width, height, radians = a
        x1, y1 = int(xyxyxyxy[i][0][0]), int(xyxyxyxy[i][0][1])
        x2, y2 = int(xyxyxyxy[i][1][0]), int(xyxyxyxy[i][1][1])
        x3, y3 = int(xyxyxyxy[i][2][0]), int(xyxyxyxy[i][2][1])
        x4, y4 = int(xyxyxyxy[i][3][0]), int(xyxyxyxy[i][3][1])

        label_id = int(classes[i])
        label_name = names[label_id]
        conf = confidence[i]

        if conf > conf_thresh:
            df_data.append({
                "x1": int(x1), "y1": int(y1),
                "x2": int(x2), "y2": int(y2),
                "x3": int(x3), "y3": int(y3),
                "x4": int(x4), "y4": int(y4),
                "centroid_x": centroid_x, "centroid_y": centroid_y,
                "width": width, "height": height, "angle": radians * 180 / 3.14159,
                "label": label_name, "confidence": conf
            })

    df = pd.DataFrame(df_data, columns=['x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4', 'centroid_x', 'centroid_y', 'width', 'height', 'angle', 'label', 'confidence'])

    if len(df) > 0:
        filtered_df = nms_classwise(df)
        if dump_images and len(df) > 0:
        # Iterate over each row in the filtered DataFrame to draw the polygons
            for index, row in filtered_df.iterrows():
            # Extract points for the current polygon
                pts = np.array([(row['x1'], row['y1']), (row['x2'], row['y2']),
                            (row['x3'], row['y3']), (row['x4'], row['y4'])], np.int32)
            # Reshape points in the form required by polylines
                pts = pts.reshape((-1, 1, 2))

            # Use specified color or default if not specified
                color = color_codes.get(row['label'], color_codes["default"])

            # Draw the polygon on the image
                cv2.polylines(im_copy, [pts], isClosed=True, color=color, thickness=2)

        # Save the modified image
            output_image_path = os.path.join('results', os.path.basename(image_path))
            cv2.imwrite(output_image_path, im_copy)
    else:
        return df, 0
    final_rotation = 0
    yolo_data = []
    if estimate_rotation:
        has_rotation, median_rotation = get_symmetrical_angle_ranges_with_median(df)
        final_rotation = infer_document_orientation(image_path, filtered_df, median_rotation)
    return filtered_df, final_rotation


def infer_document_orientation(image_path, filtered_df, median_rotation):
    # Load the image to get its dimensions
    document_width, document_height = Image.open(image_path).size

    # Function to infer orientation based on sorted elements (logos or headers)
    def infer_orientation_from_elements(elements):
        if len(elements) > 0:
            # Sort elements first by confidence (descending) and then by centroid_y (ascending)
            valid_element = elements.sort_values(by=['confidence', 'centroid_y'], ascending=[False, True]).iloc[0]

            # Define margins and alignment tolerance
            left_margin = document_width * 0.1
            alignment_tolerance = document_width * 0.1  # 10% of document width

            # Check alignment and position based on median_rotation for orientation clues
            if median_rotation < 25:  # Potential orientations: 0 or 180 degrees
                if valid_element['centroid_y'] < document_height / 3:
                    return median_rotation  # Element at top, suggesting 0 degrees rotation
                elif valid_element['centroid_y'] > 2 * document_height / 3:
                    return 180 - median_rotation  # Element at bottom, suggesting 180 degrees rotation
            elif median_rotation > 75:
                if valid_element['centroid_x'] < document_width / 3:
                    return 90  # Counter-clockwise 90 degrees
                elif valid_element['centroid_x'] > 2 * document_width / 3:
                    return -90  # Clockwise 90 degrees
        return -1  # Default return value if no orientation is inferred

    # Try with logos first
    logos = filtered_df[filtered_df['label'] == 'logo']
    orientation = infer_orientation_from_elements(logos)

    # If no conclusive orientation from logos, try with headers
    if orientation == -1:
        headers = filtered_df[filtered_df['label'] == 'header']
        orientation = infer_orientation_from_elements(headers)

    return orientation
def get_document_roi(yolo_boxes_df, image_height, image_width):
    # Initialize default crop coordinates to image boundaries
    x1, x2 = 0, image_width
    y1, y2 = 0, image_height

    # Update x1 and x2 based on 'table' detections
    table_df = yolo_boxes_df[yolo_boxes_df['label'] == 'table']
    if not table_df.empty:
        x1 = table_df[['x1', 'x2', 'x3', 'x4']].min().min()
        x2 = table_df[['x1', 'x2', 'x3', 'x4']].max().max()

    # Update y1 based on 'stamp' or 'header' detections
    logo_header_df = yolo_boxes_df[yolo_boxes_df['label'].isin(['logo', 'header'])]
    if not logo_header_df.empty:
        y1 = logo_header_df[['y1', 'y2', 'y3', 'y4']].min().min()

    # Update y2 based on 'stamp' detections
    stamp_df = yolo_boxes_df[yolo_boxes_df['label'] == 'stamp']
    if not stamp_df.empty:
        y2 = stamp_df[['y1', 'y2', 'y3', 'y4']].max().max()


    # Apply a margin around the detected ROI
    MARGIN = 0.15
    x1 = max(int(x1 - MARGIN * (x2 - x1)), 0)
    x2 = min(int(x2 + MARGIN * (x2 - x1)), image_width)
    y1 = max(int(y1 - MARGIN * (y2 - y1)), 0)
    y2 = min(int(y2 + MARGIN * (y2 - y1)), image_height)

    if x1 >= image_width //2:
        x1 = 0
    if y1 > image_height //4:
        y1 = 0
    if x2 < image_width //2:
        x2 = image_width
    if y2 < 3*image_height //4:
        y2 = image_height
    return x1, y1, x2, y2
def rotate_image(image_pil, angle):
    # Convert PIL image to OpenCV format
    image = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)

    # Get the dimensions of the image
    h, w = image.shape[:2]

    # Calculate the center of the image
    center = (w / 2, h / 2)

    # Calculate the rotation matrix
    M = cv2.getRotationMatrix2D(center, angle, 1.0)

    # Calculate the cosine and sine of the rotation angle (in radians)
    angle_rad = np.deg2rad(angle)
    cos_a = np.abs(np.cos(angle_rad))
    sin_a = np.abs(np.sin(angle_rad))

    # Compute the new bounding dimensions of the image
    new_w = int((h * sin_a) + (w * cos_a))
    new_h = int((h * cos_a) + (w * sin_a))

    # Adjust the rotation matrix to consider translation
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]

    # Perform the actual rotation

    rotated = cv2.warpAffine(image, M, (new_w, new_h))

    # Convert the OpenCV image back to PIL format
    rotated_pil = Image.fromarray(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB))

    # Return the PIL image
    return rotated_pil

def crop_document(image_path):
    document_image = Image.open(image_path)
    document_width, document_height = document_image.size
    df, rotation = get_yolo_boxes(image_path, document_image)
    # Rotation handling
    if rotation == -1:
        return -1, document_image.convert('RGB')
    elif rotation == -90 or rotation == 90:
        # document_image = document_image.rotate(-rotation, expand=True)
        document_image = rotate_image(document_image, -rotation)
        df, _ = get_yolo_boxes(image_path, document_image, estimate_rotation = False)
    elif rotation > 150:
        # document_image = document_image.rotate(rotation, expand=True)
        document_image = rotate_image(document_image, rotation)
        df, _ = get_yolo_boxes(image_path, document_image, estimate_rotation = False)
        # Rotate image 180 degrees

    # After rotation, adjust dimensions if necessary
    if rotation in [-90, 90, 180]:
        document_width, document_height = document_image.size
    # Filter for relevant labels before passing to get_document_roi
    relevant_labels = ['stamp', 'table', 'logo', 'header']
    relevant_df = df[df['label'].isin(relevant_labels)]

    # Use get_document_roi to calculate the crop boundaries if relevant detections are present
    if not relevant_df.empty:
        crop_x1, crop_y1, crop_x2, crop_y2 = get_document_roi(relevant_df, document_height, document_width)
        cropped_img = document_image.crop((crop_x1, crop_y1, crop_x2, crop_y2))
    else:
        # If no relevant detections, use the full image
        crop_x1, crop_y1, crop_x2, crop_y2 = 0, 0, document_width, document_height
        cropped_img = document_image.crop((crop_x1, crop_y1, crop_x2, crop_y2))
        return 0, cropped_img.convert('RGB')
    # Crop and return the image

    return 0, cropped_img.convert('RGB')
def conv3x3(in_channels, out_channels, kernel_size, stride=1):
    return nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        stride=stride,
        padding=kernel_size // 2,
    )


def dilated_conv_bn_act(in_channels, out_channels, act_fn, BatchNorm, dilation):
    model = nn.Sequential(
        nn.Conv2d(
            in_channels,
            out_channels,
            bias=False,
            kernel_size=3,
            stride=1,
            padding=dilation,
            dilation=dilation,
        ),
        BatchNorm(out_channels),
        act_fn,
    )
    return model


def dilated_conv(in_channels, out_channels, kernel_size, dilation, stride=1):
    model = nn.Sequential(
        nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=dilation * (kernel_size // 2),
            dilation=dilation,
        )
    )
    return model


class ResidualBlockWithDilation(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        BatchNorm,
        kernel_size,
        stride=1,
        downsample=None,
        is_activation=True,
        is_top=False,
    ):
        super(ResidualBlockWithDilation, self).__init__()
        self.stride = stride
        self.downsample = downsample
        self.is_activation = is_activation
        self.is_top = is_top
        if self.stride != 1 or self.is_top:
            self.conv1 = conv3x3(in_channels, out_channels, kernel_size, self.stride)
            self.conv2 = conv3x3(out_channels, out_channels, kernel_size)
        else:
            self.conv1 = dilated_conv(in_channels, out_channels, kernel_size, dilation=3)
            self.conv2 = dilated_conv(out_channels, out_channels, kernel_size, dilation=3)

        self.bn1 = BatchNorm(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.bn2 = BatchNorm(out_channels)

    def forward(self, x):
        residual = x
        if self.downsample is not None:
            residual = self.downsample(x)

        out1 = self.relu(self.bn1(self.conv1(x)))
        out2 = self.bn2(self.conv2(out1))

        out2 += residual
        out = self.relu(out2)
        return out


class ResnetStraight(nn.Module):
    def __init__(
        self,
        num_filter,
        map_num,
        BatchNorm,
        block_nums=[3, 4, 6, 3],
        block=ResidualBlockWithDilation,
        kernel_size=5,
        stride=[1, 1, 2, 2],
    ):
        super(ResnetStraight, self).__init__()
        self.in_channels = num_filter * map_num[0]
        self.stride = stride
        self.relu = nn.ReLU(inplace=True)
        self.block_nums = block_nums
        self.kernel_size = kernel_size

        self.layer1 = self.blocklayer(
            block,
            num_filter * map_num[0],
            self.block_nums[0],
            BatchNorm,
            kernel_size=self.kernel_size,
            stride=self.stride[0],
        )
        self.layer2 = self.blocklayer(
            block,
            num_filter * map_num[1],
            self.block_nums[1],
            BatchNorm,
            kernel_size=self.kernel_size,
            stride=self.stride[1],
        )
        self.layer3 = self.blocklayer(
            block,
            num_filter * map_num[2],
            self.block_nums[2],
            BatchNorm,
            kernel_size=self.kernel_size,
            stride=self.stride[2],
        )

    def blocklayer(self, block, out_channels, block_nums, BatchNorm, kernel_size, stride=1):
        downsample = None
        if (stride != 1) or (self.in_channels != out_channels):
            downsample = nn.Sequential(
                conv3x3(
                    self.in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    stride=stride,
                ),
                BatchNorm(out_channels),
            )

        layers = []
        layers.append(
            block(
                self.in_channels,
                out_channels,
                BatchNorm,
                kernel_size,
                stride,
                downsample,
                is_top=True,
            )
        )
        self.in_channels = out_channels
        for i in range(1, block_nums):
            layers.append(
                block(
                    out_channels,
                    out_channels,
                    BatchNorm,
                    kernel_size,
                    is_activation=True,
                    is_top=False,
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        out1 = self.layer1(x)
        out2 = self.layer2(out1)
        out3 = self.layer3(out2)
        return out3


class UVDocnet(nn.Module):
    def __init__(self, num_filter, kernel_size=5):
        super(UVDocnet, self).__init__()
        self.num_filter = num_filter
        self.in_channels = 3
        self.kernel_size = kernel_size
        self.stride = [1, 2, 2, 2]

        BatchNorm = nn.BatchNorm2d
        act_fn = nn.ReLU(inplace=True)
        map_num = [1, 2, 4, 8, 16]

        self.resnet_head = nn.Sequential(
            nn.Conv2d(
                self.in_channels,
                self.num_filter * map_num[0],
                bias=False,
                kernel_size=self.kernel_size,
                stride=2,
                padding=self.kernel_size // 2,
            ),
            BatchNorm(self.num_filter * map_num[0]),
            act_fn,
            nn.Conv2d(
                self.num_filter * map_num[0],
                self.num_filter * map_num[0],
                bias=False,
                kernel_size=self.kernel_size,
                stride=2,
                padding=self.kernel_size // 2,
            ),
            BatchNorm(self.num_filter * map_num[0]),
            act_fn,
        )

        self.resnet_down = ResnetStraight(
            self.num_filter,
            map_num,
            BatchNorm,
            block_nums=[3, 4, 6, 3],
            block=ResidualBlockWithDilation,
            kernel_size=self.kernel_size,
            stride=self.stride,
        )

        map_num_i = 2
        self.bridge_1 = nn.Sequential(
            dilated_conv_bn_act(
                self.num_filter * map_num[map_num_i],
                self.num_filter * map_num[map_num_i],
                act_fn,
                BatchNorm,
                dilation=1,
            )
        )

        self.bridge_2 = nn.Sequential(
            dilated_conv_bn_act(
                self.num_filter * map_num[map_num_i],
                self.num_filter * map_num[map_num_i],
                act_fn,
                BatchNorm,
                dilation=2,
            )
        )

        self.bridge_3 = nn.Sequential(
            dilated_conv_bn_act(
                self.num_filter * map_num[map_num_i],
                self.num_filter * map_num[map_num_i],
                act_fn,
                BatchNorm,
                dilation=5,
            )
        )

        self.bridge_4 = nn.Sequential(
            *[
                dilated_conv_bn_act(
                    self.num_filter * map_num[map_num_i],
                    self.num_filter * map_num[map_num_i],
                    act_fn,
                    BatchNorm,
                    dilation=d,
                )
                for d in [8, 3, 2]
            ]
        )

        self.bridge_5 = nn.Sequential(
            *[
                dilated_conv_bn_act(
                    self.num_filter * map_num[map_num_i],
                    self.num_filter * map_num[map_num_i],
                    act_fn,
                    BatchNorm,
                    dilation=d,
                )
                for d in [12, 7, 4]
            ]
        )

        self.bridge_6 = nn.Sequential(
            *[
                dilated_conv_bn_act(
                    self.num_filter * map_num[map_num_i],
                    self.num_filter * map_num[map_num_i],
                    act_fn,
                    BatchNorm,
                    dilation=d,
                )
                for d in [18, 12, 6]
            ]
        )

        self.bridge_concat = nn.Sequential(
            nn.Conv2d(
                self.num_filter * map_num[map_num_i] * 6,
                self.num_filter * map_num[2],
                bias=False,
                kernel_size=1,
                stride=1,
                padding=0,
            ),
            BatchNorm(self.num_filter * map_num[2]),
            act_fn,
        )

        self.out_point_positions2D = nn.Sequential(
            nn.Conv2d(
                self.num_filter * map_num[2],
                self.num_filter * map_num[0],
                bias=False,
                kernel_size=self.kernel_size,
                stride=1,
                padding=self.kernel_size // 2,
                padding_mode="reflect",
            ),
            BatchNorm(self.num_filter * map_num[0]),
            nn.PReLU(),
            nn.Conv2d(
                self.num_filter * map_num[0],
                2,
                kernel_size=self.kernel_size,
                stride=1,
                padding=self.kernel_size // 2,
                padding_mode="reflect",
            ),
        )

        self.out_point_positions3D = nn.Sequential(
            nn.Conv2d(
                self.num_filter * map_num[2],
                self.num_filter * map_num[0],
                bias=False,
                kernel_size=self.kernel_size,
                stride=1,
                padding=self.kernel_size // 2,
                padding_mode="reflect",
            ),
            BatchNorm(self.num_filter * map_num[0]),
            nn.PReLU(),
            nn.Conv2d(
                self.num_filter * map_num[0],
                3,
                kernel_size=self.kernel_size,
                stride=1,
                padding=self.kernel_size // 2,
                padding_mode="reflect",
            ),
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_normal_(m.weight, gain=0.2)
            if isinstance(m, nn.ConvTranspose2d):
                assert m.kernel_size[0] == m.kernel_size[1]
                nn.init.xavier_normal_(m.weight, gain=0.2)

    def forward(self, x):
        resnet_head = self.resnet_head(x)
        resnet_down = self.resnet_down(resnet_head)
        bridge_1 = self.bridge_1(resnet_down)
        bridge_2 = self.bridge_2(resnet_down)
        bridge_3 = self.bridge_3(resnet_down)
        bridge_4 = self.bridge_4(resnet_down)
        bridge_5 = self.bridge_5(resnet_down)
        bridge_6 = self.bridge_6(resnet_down)
        bridge_concat = torch.cat([bridge_1, bridge_2, bridge_3, bridge_4, bridge_5, bridge_6], dim=1)
        bridge = self.bridge_concat(bridge_concat)

        out_point_positions2D = self.out_point_positions2D(bridge)
        out_point_positions3D = self.out_point_positions3D(bridge)

        return out_point_positions2D, out_point_positions3D
def bilinear_unwarping(warped_img, point_positions, img_size):
    upsampled_grid = F.interpolate(
        point_positions, size=(img_size[1], img_size[0]), mode="bilinear", align_corners=True
    )
    unwarped_img = F.grid_sample(warped_img, upsampled_grid.transpose(1, 2).transpose(2, 3), align_corners=True)

    return unwarped_img
def load_rec_model(ckpt_path):
    """
    Load UVDocnet model.
    """
    # device = torch.device("cpu")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    model = UVDocnet(num_filter=32, kernel_size=5)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.to(device)
    model.eval()
    return model

def unwarp_img(model, device, image, final_doc_size):
    IMG_SIZE = [488, 712]
    model.eval()
    img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32) / 255
    inp = torch.from_numpy(cv2.resize(img, IMG_SIZE).transpose(2, 0, 1)).unsqueeze(0)

    # Make prediction
    inp = inp.to(device)
    point_positions2D, _ = model(inp)

    unwarped = bilinear_unwarping(
        warped_img=torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).to(device),
        point_positions=torch.unsqueeze(point_positions2D[0], dim=0),
        img_size=tuple(final_doc_size),
    )
    unwarped = (unwarped[0].detach().cpu().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)

    # Save result
    unwarped_BGR = cv2.cvtColor(unwarped, cv2.COLOR_RGB2BGR)

    return unwarped_BGR
def check_blur_fourier(image_path):
    image = cv2.imread(image_path, 0)  # Read in grayscale
    if image is None:
        return "Image not found or path is incorrect."

    # Compute the FFT of the image
    f = np.fft.fft2(image)
    fshift = np.fft.fftshift(f)
    magnitude_spectrum = 20*np.log(np.abs(fshift))
    threshold = 130  # This is a hypothetical threshold and needs to be tuned
    mean_val = np.mean(magnitude_spectrum)
    if mean_val < threshold:
        return False  # Likely blurred
    else:
        return True  # Likely not blurred
def document_rectification(image_file_path, final_doc_size=(1200, 1700), save_img=True,
                           path_to_save='./rectified/', unwrap = True):
    valid, cropped_img = crop_document(image_file_path)
    # if not check_blur_fourier(image_file_path):
    #     print('failed ::: ', image_file_path)
    #     return [], []
    if valid == -1:
        print('failed ::: ', image_file_path)
        return [], []
    cropped_img = np.array(cropped_img)[:, :, ::-1]
    if unwrap:
        # unwarped_img = unwarp_img(rec_model, 'cpu', cropped_img, final_doc_size)
        unwarped_img = unwarp_img(rec_model, 'cuda', cropped_img, final_doc_size)

    else:
        unwarped_img = cropped_img

    if save_img:
        output_image_path = os.path.join(path_to_save, os.path.basename(image_file_path))
        cv2.imwrite(output_image_path, unwarped_img)

    return cropped_img, unwarped_img

det_model = YOLO('../models/rotation_22may.pt')
rec_model = load_rec_model('../models/rect_model.pkl')


In [7]:
import pandas as pd

def get_iou_inside(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    unionArea = min(boxAArea, boxBArea)
    if unionArea == 0:
        return 0
    iou = interArea / float(unionArea)
    return iou


def get_stamp_locations(df, width, height):
    stamps = df[df['label'] == 'stamp'].copy()
    stamps = stamps[stamps['confidence'] > .4]
    tables = df[df['label'] == 'table'].copy()
    signatures = df[df['label'] == 'signature'].copy()
    # signatures = signatures[signatures['confidence'] > .5]
    stamp_middle = []
    stamp_bottom_middle = []
    stamp_bottom_left = []
    selected_mid_stamp  = None
    bottom_middle_indices = []
    largest_table = tables.sort_values(by='height', ascending=False).iloc[0] if not tables.empty else None
    if largest_table is not None:
        table_x1, table_y1, table_x2, table_y2 = min(largest_table['x1'], largest_table['x2'], largest_table['x3'], largest_table['x4']), min(largest_table['y1'], largest_table['y2'], largest_table['y3'], largest_table['y4']), max(largest_table['x1'], largest_table['x2'], largest_table['x3'], largest_table['x4']), max(largest_table['y1'], largest_table['y2'], largest_table['y3'], largest_table['y4'])
        mid_stamps = stamps[
            (stamps['centroid_x'] >= table_x1) & (stamps['centroid_x'] <= table_x2) &
            (stamps['centroid_y'] >= table_y1 + (table_y2 - table_y2) * .2 ) & (stamps['centroid_y'] <= table_y2 - (table_y2 - table_y2) * .2)
        ]
        if not mid_stamps.empty:
            selected_mid_stamp = mid_stamps.sort_values(by='confidence', ascending=False).iloc[0]
            stamp_middle.append(selected_mid_stamp.to_dict())
    if not stamps.empty and (largest_table is not None):
        try:
            selected_mid_stamp_x1 = min(selected_mid_stamp['x1'], selected_mid_stamp['x2'], selected_mid_stamp['x3'], selected_mid_stamp['x4'])
            selected_mid_stamp_x2 = max(selected_mid_stamp['x1'], selected_mid_stamp['x2'], selected_mid_stamp['x3'], selected_mid_stamp['x4'])
            bottom_middle_stamps = stamps[
                (stamps['centroid_y'] > table_y2) &
                (stamps['centroid_x'] > selected_mid_stamp_x1) & (stamps['centroid_x'] < selected_mid_stamp_x2)]
        except:
            bottom_middle_stamps = stamps[(stamps['centroid_y'] > table_y2) & (stamps['centroid_x'] > width/3) & (stamps['centroid_x'] < 2*width/3)]
        if not bottom_middle_stamps.empty:
            selected_bottom_mid_stamp = bottom_middle_stamps.sort_values(by='centroid_x', ascending=False).iloc[0]
            stamp_bottom_middle.append(selected_bottom_mid_stamp.to_dict())
            bottom_middle_indices.append(selected_bottom_mid_stamp.name)

    # Find bottom left stamps
    if not stamps.empty and (largest_table is not None):
        rem_stamp = stamps[~stamps.index.isin(bottom_middle_indices)]
        bottom_left_stamps = rem_stamp[(rem_stamp['centroid_y'] > table_y2)]
        best_stamp = None
        best_sign = None
        for stamp in bottom_left_stamps.itertuples():
            max_iou = 0
            stamp_box = [
                min(stamp.x1, stamp.x2, stamp.x3, stamp.x4),
                min(stamp.y1, stamp.y2, stamp.y3, stamp.y4),
                max(stamp.x1, stamp.x2, stamp.x3, stamp.x4),
                max(stamp.y1, stamp.y2, stamp.y3, stamp.y4)]
            for sign in signatures.itertuples():
                sign_box = [
                min(sign.x1, sign.x2, sign.x3, sign.x4),
                min(sign.y1, sign.y2, sign.y3, sign.y4),
                max(sign.x1, sign.x2, sign.x3, sign.x4),
                max(sign.y1, sign.y2, sign.y3, sign.y4)]
                iou =  get_iou(stamp_box, sign_box)
                if iou >= max_iou:
                    max_iou = iou
                    best_stamp = stamp
                    best_sign = sign
            if best_sign and best_stamp:
                if max_iou >= 0 and (best_sign.height * best_sign.width) < (best_stamp.height * best_stamp.width):
                    #print(max_iou, best_sign.height/best_stamp.height)
                    if max_iou < .5:
                        if best_stamp:
                            stamp_bottom_left.append({
                                'x1': best_stamp.x1, 'y1': best_stamp.y1,'x2': best_stamp.x2, 'y2': best_stamp.y2,'centroid_x': best_stamp.centroid_x,'centroid_y': best_stamp.centroid_y,'width': getattr(best_stamp, 'width', 0),'height': getattr(best_stamp, 'height', 0),'angle': getattr(best_stamp, 'angle', 0),'label': 'stamp','confidence': best_stamp.confidence
                            })
                    elif max_iou >=.5 and best_sign.height/best_stamp.height <=.6:
                        #print('max_iou >=.5 and best_sign.height/best_stamp.height <=.7:')
                        stamp_bottom_left.append({
                                'x1': best_stamp.x1, 'y1': best_stamp.y1,'x2': best_stamp.x2, 'y2': best_stamp.y2,'centroid_x': best_stamp.centroid_x,'centroid_y': best_stamp.centroid_y,'width': getattr(best_stamp, 'width', 0),'height': getattr(best_stamp, 'height', 0),'angle': getattr(best_stamp, 'angle', 0),'label': 'stamp','confidence': best_stamp.confidence
                            })
                    elif max_iou >=.5 and best_sign.height/best_stamp.height >=.6 and best_stamp.confidence > .85:
                        #print('max_iou >=.5 and best_sign.height/best_stamp.height >=.6 and best_stamp.confidence > .75:')
                        stamp_bottom_left.append({
                                'x1': best_stamp.x1, 'y1': best_stamp.y1,'x2': best_stamp.x2, 'y2': best_stamp.y2,'centroid_x': best_stamp.centroid_x,'centroid_y': best_stamp.centroid_y,'width': getattr(best_stamp, 'width', 0),'height': getattr(best_stamp, 'height', 0),'angle': getattr(best_stamp, 'angle', 0),'label': 'stamp','confidence': best_stamp.confidence
                            })
            else:
                stamp_bottom_left = bottom_left_stamps
    return pd.DataFrame(stamp_middle), pd.DataFrame(stamp_bottom_middle), pd.DataFrame(stamp_bottom_left)

def get_signature_locations(df, width, height):
    signatures = df[df['label'] == 'signature'].copy()
    tables = df[df['label'] == 'table'].copy()
    paragraphs = df[df['label'] == 'paragraph'].copy()
    stamps = df[df['label'] == 'stamp'].copy()
    columns = df[df['label'] == 'column'].copy()
    largest_table = tables.sort_values(by='height', ascending=False).iloc[0] if not tables.empty else None
    valid_signatures_in_agent_column = []
    valid_signature_below_document = []
    valid_signature_in_paragraph = []
    if largest_table is not None:
        table_bounds = [
            min(largest_table['x1'], largest_table['x2'], largest_table['x3'], largest_table['x4']),
            min(largest_table['y1'], largest_table['y2'], largest_table['y3'], largest_table['y4']),
            max(largest_table['x1'], largest_table['x2'], largest_table['x3'], largest_table['x4']),
            max(largest_table['y1'], largest_table['y2'], largest_table['y3'], largest_table['y4'])
        ]
        best_col = None
        best_col_centroid = -1
        for col in columns.itertuples():
            column_bound = [
                min(col.x1, col.x2, col.x3, col.x4),
                min(col.y1, col.y2, col.y3, col.y4),
                max(col.x1, col.x2, col.x3, col.x4),
                max(col.y1, col.y2, col.y3, col.y4)]
            if (get_iou_inside(table_bounds, column_bound) >.85) and (col.centroid_x > (table_bounds[0] + table_bounds[2])/2):
                if col.centroid_x > best_col_centroid:
                    best_col_centroid = col.centroid_x
                    best_col = column_bound
        right_most_col = best_col
        right_half_start = (table_bounds[0] + table_bounds[2]) /2
        if best_col:
            right_half_start = best_col[0]
            for _, sig in signatures.iterrows():
                if sig['centroid_x'] > right_half_start and sig['centroid_y'] > table_bounds[1] and sig['centroid_y'] < table_bounds[3]:
                    valid_signatures_in_agent_column.append(sig.to_dict())
    else:
        return pd.DataFrame([]), pd.DataFrame([]), pd.DataFrame([])

    most_confident_paragraph = paragraphs.sort_values(by='confidence', ascending=False).iloc[0] if not paragraphs.empty else None
    if most_confident_paragraph is not None:
        para_centroid_x = most_confident_paragraph['centroid_x']
        para_centroid_y =  most_confident_paragraph['centroid_y']
        para_bbox = [min(most_confident_paragraph.x1, most_confident_paragraph.x2, most_confident_paragraph.x3, most_confident_paragraph.x4),
                    min(most_confident_paragraph.y1, most_confident_paragraph.y2, most_confident_paragraph.y3, most_confident_paragraph.y4),
                    max(most_confident_paragraph.x1, most_confident_paragraph.x2, most_confident_paragraph.x3, most_confident_paragraph.x4),
                    max(most_confident_paragraph.y1, most_confident_paragraph.y2, most_confident_paragraph.y3, most_confident_paragraph.y4)]
    else:
        try:
            bottom_signatures = signatures[(signatures['centroid_y'] > table_bounds[3]) & (signatures['centroid_x'] < width * 0.5)]
            return pd.DataFrame(valid_signatures_in_agent_column), pd.DataFrame(bottom_signatures), pd.DataFrame([])
        except:
            return pd.DataFrame(valid_signatures_in_agent_column), pd.DataFrame([]), pd.DataFrame([])
    bottom_signatures = signatures[(signatures['centroid_y'] > min(table_bounds[3], para_bbox[1])) & (signatures['centroid_x'] < width * 0.5)]
    best_sig = None
    best_sig_below = None
    for sig in bottom_signatures.itertuples():
        max_iou = 0
        for _, stamp in stamps.iterrows():
            stamp_bbox = [
                        min(stamp['x1'], stamp['x2'], stamp['x3'], stamp['x4']),
                        min(stamp['y1'], stamp['y2'], stamp['y3'], stamp['y4']),
                        max(stamp['x1'], stamp['x2'], stamp['x3'], stamp['x4']),
                        max(stamp['y1'], stamp['y2'], stamp['y3'], stamp['y4'])]
            sig_bbox = [
                    min(sig.x1, sig.x2, sig.x3, sig.x4),
                    min(sig.y1, sig.y2, sig.y3, sig.y4),
                    max(sig.x1, sig.x2, sig.x3, sig.x4),
                    max(sig.y1, sig.y2, sig.y3, sig.y4)]
            iou = get_iou_inside(sig_bbox, stamp_bbox)
            if iou > max_iou:
                max_iou = iou
                best_sig = sig._asdict()
        if best_sig and max_iou < .7:
            if most_confident_paragraph is not None:
                best_sig_bbox = [
                        min(best_sig['x1'], best_sig['x2'], best_sig['x3'], best_sig['x4']),
                        min(best_sig['y1'], best_sig['y2'], best_sig['y3'], best_sig['y4']),
                        max(best_sig['x1'], best_sig['x2'], best_sig['x3'], best_sig['x4']),
                        max(best_sig['y1'], best_sig['y2'], best_sig['y3'], best_sig['y4'])]
                if get_iou_inside(best_sig_bbox, para_bbox) <.8:
                    valid_signature_below_document.append(best_sig)
                    bottom_signatures = bottom_signatures[bottom_signatures.index != best_sig['Index']]
            else:
                valid_signature_below_document.append(best_sig)
                bottom_signatures = bottom_signatures[bottom_signatures.index != best_sig['Index']]
        elif best_sig and max_iou > .7:
            valid_signature_below_document.append(best_sig)
            bottom_signatures = bottom_signatures[bottom_signatures.index != best_sig['Index']]
    best_sign = None
    best_sig_below = []
    for sig in bottom_signatures.itertuples():
        max_iou = 0
        max_iou_matched = 0
        best_sig_para = None
        sig_bbox = [
                        min(sig.x1, sig.x2, sig.x3, sig.x4),
                        min(sig.y1, sig.y2, sig.y3, sig.y4),
                        max(sig.x1, sig.x2, sig.x3, sig.x4),
                        max(sig.y1, sig.y2, sig.y3, sig.y4)]

        if sig_bbox[3] > para_bbox[3]:
            best_sig_below.append(sig._asdict())
        else:
            iou = get_iou_inside(sig_bbox, para_bbox)
            if iou >= max_iou:
                max_iou = iou
                best_sign = sig._asdict()
    if best_sign and max_iou >= .8:
        valid_signature_in_paragraph.append(best_sign)
    elif best_sign and max_iou <.8:
        valid_signature_below_document.append(best_sign)
    if len(best_sig_below) > 0:
        valid_signature_below_document = valid_signature_below_document + best_sig_below
    max_iou = 0
    sig_inside = None
    for sig in valid_signature_below_document:
        sig_bbox = [
                    min(sig['x1'], sig['x2'], sig['x3'], sig['x4']),
                    min(sig['y1'], sig['y2'], sig['y3'], sig['y4']),
                    max(sig['x1'], sig['x2'], sig['x3'], sig['x4']),
                    max(sig['y1'], sig['y2'], sig['y3'], sig['y4'])]
        iou = get_iou_inside(sig_bbox, para_bbox)
        if iou >= max_iou:
            max_iou = iou
            sig_inside = sig
    if max_iou >= .75:
        valid_signature_below_document.remove(sig_inside)
        valid_signature_in_paragraph.append(sig_inside)
    return pd.DataFrame(valid_signatures_in_agent_column), pd.DataFrame(valid_signature_below_document), pd.DataFrame(valid_signature_in_paragraph)



In [6]:
import os
from PIL import Image, ImageDraw

def draw_bounding_boxes_and_save(image_path,image, df, output_path):

    draw = ImageDraw.Draw(image)

    for _, row in df.iterrows():
        label = row['label']
        x1 = row['centroid_x'] - row['width'] / 2
        x2 = row['centroid_x'] + row['width'] / 2
        y1 = row['centroid_y'] - row['height'] / 2
        y2 = row['centroid_y'] + row['height'] / 2

        if label == 'signatures_in_agent_column':
            draw.rectangle([x1, y1, x2, y2], outline='red', width=5)
        elif label == 'signature_below_document':
            draw.rectangle([x1, y1, x2, y2], outline='green', width=5)
        elif label == 'signature_in_paragraph':
            draw.rectangle([x1, y1, x2, y2], outline='blue', width=5)
        elif label == 'stamp_middle':
            draw.rectangle([x1, y1, x2, y2], outline='#ADD8E6', width=5)
        elif label == 'stamp_bottom_middle':
            draw.rectangle([x1, y1, x2, y2], outline='#FFA500', width=5)
        elif label == 'stamp_bottom_left':
            draw.rectangle([x1, y1, x2, y2], outline='#FFC0CB', width=5)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    image.save(output_path)


In [3]:
from PIL import Image, ImageDraw
import pandas as pd
import numpy as np
import os

def process_individual_file(image_path,output_filepath):
    # Processes one image from raw model detection to final classified stamp/signature output.

    df, page_rotation = get_yolo_boxes(image_path, conf_thresh = .25, estimate_rotation = False)
    page_image = Image.open(image_path)
    width, height = page_image.size
    stamp_middle, stamp_bottom_middle, stamp_bottom_left = get_stamp_locations(df, width, height)
    signatures_in_agent_column, signature_below_document, signature_in_paragraph = get_signature_locations(df, width, height)
    # Create a list to store the DataFrames

    # Define the columns for the DataFrame
    columns = ['x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4', 'centroid_x', 'centroid_y', 'width', 'height', 'angle',  'confidence','label']
    stamp_middle['label'] = 'stamp_middle'
    stamp_bottom_left['label'] = 'stamp_bottom_left'
    stamp_bottom_middle['label'] = 'stamp_bottom_middle'
    signature_below_document['label'] = 'signature_below_document'
    signature_in_paragraph['label'] = 'signature_in_paragraph'
    signatures_in_agent_column['label'] = 'signatures_in_agent_column'
    dfs = [stamp_middle,stamp_bottom_left,stamp_bottom_middle,signature_below_document,signature_in_paragraph,signatures_in_agent_column,df]

    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df['file_path']=image_path
    file_name = os.path.basename(image_path)
    combined_df['file_name']=file_name

    combined_df.sort_values(by=['label','confidence'], ascending=False, inplace=True)
    combined_df.reset_index(inplace=True,drop=True)

    # Only use this if you want to see the document with stamp and signature bounding box draw
    # draw_bounding_boxes_and_save(image_path,page_image, combined_df, output_filepath)

    return combined_df


In [9]:
df_test=process_individual_file('./../test_images/30_26_10_008_unwrap.jpg','../test_images/stamp_test/30_26_10_008_marked.png')

In [11]:
df_test

,x1,y1,x2,y2,x3,y3,x4,y4,centroid_x,centroid_y,width,height,angle,label,confidence,Index,file_path,file_name
0,1135,1446,1137,561,66.0,559.0,65.0,1445.0,601.216919,1003.333984,1070.734741,885.256287,0.101967,table,0.947907,NaN,./../test_images/30_26_10_008_unwrap.jpg,30_26_10_008_unwrap.jpg
1,886,595,888,370,75.0,362.0,73.0,587.0,480.840851,478.850220,812.994446,224.782593,0.537907,table,0.885536,NaN,./../test_images/30_26_10_008_unwrap.jpg,30_26_10_008_unwrap.jpg
2,420,760,429,1139,822.0,1129.0,813.0,750.0,621.455444,944.965271,393.767090,379.093567,178.617657,stamp_middle,0.887796,NaN,./../test_images/30_26_10_008_unwrap.jpg,30_26_10_008_unwrap.jpg
3,573,1506,574,1689,762.0,1688.0,761.0,1505.0,667.948730,1597.756470,187.600220,182.974564,179.798899,stamp_bottom_middle,0.935241,NaN,./../test_images/30_26_10_008_unwrap.jpg,30_26_10_008_unwrap.jpg
4,447,1704,448,1465,NaN,NaN,NaN,NaN,318.695465,1583.729492,258.921387,238.914703,0.394054,stamp_bottom_left,0.891923,NaN,./../test_images/30_26_10_008_unwrap.jpg,30_26_10_008_unwrap.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,621,506,621,535,727.0,535.0,727.0,506.0,674.817505,521.124390,106.113281,28.731846,179.977713,box,0.513361,NaN,./../test_images/30_26_10_008_unwrap.jpg,30_26_10_008_unwrap.jpg
142,651,988,652,958,585.0,958.0,585.0,988.0,618.748779,973.514404,66.380653,29.825611,0.649616,box,0.401148,NaN,./../test_images/30_26_10_008_unwrap.jpg,30_26_10_008_unwrap.jpg
143,90,1467,113,1467,113.0,1441.0,91.0,1441.0,102.184578,1454.470459,25.654327,22.397751,90.859180,box,0.307476,NaN,./../test_images/30_26_10_008_unwrap.jpg,30_26_10_008_unwrap.jpg
144,197,191,198,178,61.0,169.0,60.0,183.0,129.720734,180.851807,137.000061,13.480175,3.564761,box,0.280784,NaN,./../test_images/30_26_10_008_unwrap.jpg,30_26_10_008_unwrap.jpg


In [15]:
# Load the image-processing metadata and keep only rows that successfully completed the earlier preprocessing step.
# The new unwrap_path column points to the normalized image version used in the layout workflow.
csv_file = '../test_images/batch_test/csv/images_unwrap_process.csv'
df_all=pd.read_csv(csv_file)
df_use=df_all[df_all['status']=="Success"]
df_use.reset_index(inplace=True,drop=True)
def update_file_path(file_path):
    # Converts the original cropped-image path into the corresponding unwrapped-image path.
    # Application purpose: links each metadata row to the preprocessed image version used for layout analysis.
    p1=file_path.replace('consolidated_files_jpg', 'cropped_files_jpg')
    p2=p1.replace('.jpg', '_crop.jpg')
    return p2

df_use['crop_filepath'] = df_use['jpg_filepath'].apply(update_file_path)
df_use.head()

/tmp/ipykernel_9882/2557356859.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_use['crop_filepath'] = df_use['jpg_filepath'].apply(update_file_path)


,jpg_filepath,polling_unit_code,blurred,status,crop_filepath
0,consolidated_files_jpg/state_22/lga_05/ward_10...,22/05/10/028,False,Success,cropped_files_jpg/state_22/lga_05/ward_10/22_0...
1,consolidated_files_jpg/state_22/lga_05/ward_10...,22/05/10/016,False,Success,cropped_files_jpg/state_22/lga_05/ward_10/22_0...
2,consolidated_files_jpg/state_22/lga_05/ward_10...,22/05/10/047,False,Success,cropped_files_jpg/state_22/lga_05/ward_10/22_0...
3,consolidated_files_jpg/state_22/lga_05/ward_10...,22/05/10/010,False,Success,cropped_files_jpg/state_22/lga_05/ward_10/22_0...
4,consolidated_files_jpg/state_02/lga_14/ward_04...,02/14/04/023,False,Success,cropped_files_jpg/state_02/lga_14/ward_04/02_1...


In [19]:
import multiprocessing as mp

In [20]:
# import os
# import pandas as pd
# from multiprocessing import Pool, cpu_count

# def process_image(args):
#     # Batch helper that prepares file paths, runs the single-image workflow, and records success/failure status.
#     # Application purpose: allows many result-sheet images to be processed while keeping track of failed cases.
#     index, row = args
#     image_file_path = f"../test_images/batch_test/{row['crop_filepath']}"
#     polling_unit_code = row['polling_unit_code']
#     state_code, lga_code, ward_code, polling_unit_id = polling_unit_code.split('/')
#     original_file_name = os.path.splitext(os.path.basename(image_file_path))[0]
#     new_path_stamp = f"../test_images/batch_test/stamp_jpg/state_{state_code}/lga_{lga_code}/ward_{ward_code}/{original_file_name}_stamp.jpg"

#     try:
#         # Process one form and return both its classified detections and a status flag for the batch log.
#         df_temp = process_individual_file(image_file_path, new_path_stamp)
#         return df_temp, index, 'Success'
#     except Exception as e:
#         print(f"Error processing image {image_file_path}: {e}")
#         return pd.DataFrame(), index, 'Failed'

# def main():
#     # Batch-processing entry point for applying the stamp/signature workflow across the dataset.
#     # Application purpose: resumes from existing output when available, processes remaining images, and saves updated CSV results.
#     csv_file = '../csv/layout/stamp_layout_result.csv'
#     if os.path.exists(csv_file):
#         df = pd.read_csv(csv_file)
#         print('Loading existing file.')
#     else:
#         df = df_use.copy() 
#         df.reset_index(inplace=True, drop=True)
#         df['status_stamp'] = pd.NA
#         print('Creating new DataFrame from scratch.')

#     unprocessed_rows = df[(df['status_stamp'].isna())|(df['status_stamp']=='Failed')]
#     num_processes = cpu_count()  # Utilize all available CPU cores
#     batch_size = 100  # Adjusted batch size
#     results_df = pd.DataFrame()

#     # with Pool(num_processes) as pool:
#     ctx = mp.get_context("spawn")
#     with ctx.Pool(num_processes) as pool:
#         for start in range(0, len(unprocessed_rows), batch_size):
#             results_df = pd.DataFrame()
#             end = start + batch_size
#             batch = unprocessed_rows.iloc[start:end]
#             tasks = [(idx, row) for idx, row in batch.iterrows()]
#             results = pool.map(process_image, tasks)

#             for result_df, index, status in results:
#                 df.at[index, 'status_stamp'] = status
#                 results_df = pd.concat([results_df, result_df], ignore_index=True)

#             if not results_df.empty:
#                 df_count = results_df[results_df['confidence']>0.25].groupby(by=['file_name','label'])['file_path'].count().to_frame().reset_index()
#                 df_count.to_csv(f'../csv/layout/batch_{start//batch_size }.csv', index=False)

#             df.to_csv(csv_file,index=False)  # Consider reducing frequency of saves if performance is an issue
#             print(f"Processed up to index {end}. Batch {start//batch_size } results saved.")

#     df.to_csv(csv_file, index=False)
#     print(f"Processing completed. Total files processed: {len(unprocessed_rows)}. Final results saved.")



In [12]:
import os
import pandas as pd
from multiprocessing import cpu_count
from multiprocessing.pool import ThreadPool

def process_image(args):
    # Batch helper that prepares file paths, runs the single-image workflow, and records success/failure status.
    # Application purpose: allows many result-sheet images to be processed while keeping track of failed cases.
    index, row = args
    image_file_path = f"../test_images/batch_test/{row['crop_filepath']}"
    polling_unit_code = row['polling_unit_code']
    state_code, lga_code, ward_code, polling_unit_id = polling_unit_code.split('/')
    original_file_name = os.path.splitext(os.path.basename(image_file_path))[0]
    new_path_stamp = f"../test_images/batch_test/stamp_jpg/state_{state_code}/lga_{lga_code}/ward_{ward_code}/{original_file_name}_stamp.jpg"

    try:
        # Process one form and return both its classified detections and a status flag for the batch log.
        df_temp = process_individual_file(image_file_path, new_path_stamp)
        return df_temp, index, 'Success'
    except Exception as e:
        print(f"Error processing image {image_file_path}: {e}")
        return pd.DataFrame(), index, 'Failed'

def main():
    # Batch-processing entry point for applying the stamp/signature workflow across the dataset.
    # Application purpose: resumes from existing output when available, processes remaining images, and saves updated CSV results.
    csv_file = '../csv/layout/stamp_layout_result.csv'
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        print('Loading existing file.')
    else:
        df = df_use.copy() 
        df.reset_index(inplace=True, drop=True)
        df['status_stamp'] = pd.NA
        print('Creating new DataFrame from scratch.')

    unprocessed_rows = df[(df['status_stamp'].isna())|(df['status_stamp']=='Failed')]
    num_processes = cpu_count()  # Utilize all available CPU cores
    batch_size = 100  # Adjusted batch size
    results_df = pd.DataFrame()

    with ThreadPool(num_processes) as pool:
        for start in range(0, len(unprocessed_rows), batch_size):
            results_df = pd.DataFrame()
            end = start + batch_size
            batch = unprocessed_rows.iloc[start:end]
            tasks = [(idx, row) for idx, row in batch.iterrows()]
            results = pool.map(process_image, tasks)

            for result_df, index, status in results:
                df.at[index, 'status_stamp'] = status
                results_df = pd.concat([results_df, result_df], ignore_index=True)

            if not results_df.empty:
                df_count = results_df[results_df['confidence']>0.25].groupby(by=['file_name','label'])['file_path'].count().to_frame().reset_index()
                df_count.to_csv(f'../csv/layout/batch_{start//batch_size }.csv', index=False)

            df.to_csv(csv_file,index=False)  # Consider reducing frequency of saves if performance is an issue
            print(f"Processed up to index {end}. Batch {start//batch_size } results saved.")

    df.to_csv(csv_file, index=False)
    print(f"Processing completed. Total files processed: {len(unprocessed_rows)}. Final results saved.")



In [13]:
main()

Loading existing file.
Processed up to index 100. Batch 0 results saved.
Processing completed. Total files processed: 22. Final results saved.


In [19]:

# concat all batch run result to get the result dataframe with coordinates for elements


import pandas as pd
import glob

# Path to the directory containing CSV files
folder_path = '../csv/layout/'

# Glob pattern to match the files
file_pattern = f"{folder_path}/batch*.csv"

# Use glob to find all files matching the pattern
file_list = glob.glob(file_pattern)

# Read each file into a DataFrame and concatenate into one DataFrame
combined_df = pd.concat((pd.read_csv(file) for file in file_list), ignore_index=True)



In [21]:
# count the number of elements on each election paper
df_pivot=combined_df.pivot_table(index='file_name', columns='label', values='file_path', aggfunc='sum')
df_pivot.reset_index(inplace=True)
df_pivot.fillna(0, inplace=True)

In [23]:
def filter(df):
    # Finds images that may need review based on unexpected counts of detected layout elements.

    # These are outlier documents that likely are not in the form of EC84
  f1=df['box']>200
  f2=df['column']>10
  f3=df['kv']<2
  f5=df['paragraph']<1
  f13=df['table']<2

# These are the documents missing stamps or signatures
  f6=df['signature_below_document']==0
  f9=df['signatures_in_agent_column']==0
  f8=df['signature_in_paragraph']==0
  f10=df['stamp_bottom_left']==0
  f11=df['stamp_bottom_middle']==0
  f12=df['stamp_middle']==0

  df_outlier=df[(f1|f2|f3|f5|f13)]
  df_missing=df[(f6|f9|f8|f10|f11|f12)]
  df_outlier.reset_index(inplace=True,drop=True)
  df_missing.reset_index(inplace=True,drop=True)
  return df_missing,df_outlier


In [24]:
df_missing_stampsig=filter(df_pivot)[0]
df_outlier_doc=filter(df_pivot)[1]



In [26]:
# These problematic documents are then sent for human verification to check which kind of paper it is & if elements are missin g

label,file_name,box,column,figure,header,kv,logo,paragraph,signature,signature_below_document,signature_in_paragraph,signatures_in_agent_column,stamp,stamp_bottom_left,stamp_bottom_middle,stamp_middle,table
0,01_01_01_002_crop.jpg,32.0,2.0,2.0,3.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0,2.0,0.0,1.0,0.0,1.0
1,01_01_01_003_crop.jpg,75.0,4.0,0.0,0.0,1.0,1.0,1.0,4.0,0.0,2.0,2.0,2.0,1.0,1.0,0.0,1.0
2,01_01_01_015_crop.jpg,76.0,5.0,0.0,3.0,5.0,1.0,1.0,12.0,1.0,1.0,9.0,2.0,1.0,1.0,0.0,1.0
3,01_01_03_004_crop.jpg,94.0,5.0,0.0,3.0,5.0,1.0,1.0,9.0,1.0,1.0,7.0,2.0,1.0,1.0,0.0,1.0
4,01_01_06_030_crop.jpg,114.0,7.0,0.0,5.0,5.0,1.0,2.0,6.0,0.0,2.0,4.0,3.0,0.0,1.0,1.0,2.0
5,01_04_03_017_crop.jpg,11.0,0.0,4.0,3.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,01_05_08_007_crop.jpg,115.0,7.0,0.0,3.0,5.0,1.0,1.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0
7,01_10_10_013_crop.jpg,102.0,7.0,0.0,3.0,4.0,1.0,1.0,7.0,0.0,1.0,6.0,2.0,1.0,0.0,1.0,2.0
8,01_10_10_019_crop.jpg,101.0,7.0,0.0,3.0,4.0,1.0,1.0,5.0,1.0,0.0,4.0,2.0,1.0,0.0,1.0,2.0
9,01_10_10_021_crop.jpg,103.0,7.0,0.0,3.0,5.0,1.0,1.0,6.0,1.0,1.0,4.0,2.0,1.0,0.0,1.0,2.0
